In [ ]:
import os
import csv
import hashlib
from datetime import datetime
from flask import Flask, request, redirect, url_for, session, Response, jsonify, render_template_string

app = Flask(__name__)
app.secret_key = 'ucc-phishing-2026-group-project-super-secure-key-98765'
app.config['SESSION_TYPE'] = 'filesystem'
app.config['DEBUG'] = False
app.config['PERMANENT_SESSION_LIFETIME'] = 3600  # 1 hour

# CSV file for storing responses (backup)
CSV_FILE = 'phishing_survey_responses_comprehensive.csv'

# Google Sheets Configuration
# ===========================
GOOGLE_SHEETS_ENABLED = True  # We'll set this to True after setup

# We'll add Google Sheets integration after getting your credentials

def ensure_csv_headers():
    """Create CSV file with comprehensive headers"""
    if not os.path.exists(CSV_FILE):
        headers = [
            # Metadata
            'Timestamp', 'Response_ID', 'Completion_Time_Seconds', 'IP_Address_Hash',
            
            # Section A: Demographics
            'Gender', 'Age_Range', 'Level_of_Study', 'College', 'Program_of_Study', 
            'Student_Group_Classification', 'STEM_Confidence',
            
            # Section B: Digital Literacy & Experience
            'Email_Usage_Frequency', 'Email_Accounts_Count', 'Primary_Email_Provider',
            'Mobile_Banking_Frequency', 'Mobile_Money_Usage', 'Online_Shopping_Frequency',
            'Social_Media_Platforms_Count', 'Primary_Social_Media', 'Years_of_Internet_Experience',
            'Received_Suspicious_Messages', 'Suspicious_Message_Frequency',
            'Suspicious_Message_Channels', 'Previous_Cybersecurity_Training',
            'Training_Types', 'Self_Rated_Digital_Literacy',
            
            # Section C: Phishing Knowledge Assessment
            'Familiar_With_Phishing', 'Phishing_Definition_Correct',
            'Phishing_Indicators_Selected', 'Password_Statement_Correct',
            'URL_Hovering_Knowledge', 'HTTPS_Knowledge', 'Padlock_Icon_Knowledge',
            'Email_Header_Knowledge', 'Spear_Phishing_Awareness', 'Smishing_Awareness',
            'Vishing_Awareness', 'Phishing_Statistics_Knowledge', 'UCC_Phishing_Reports_Awareness',
            'Knowledge_Score', 'Knowledge_Score_Percentage',
            
            # Section D: Behavioral Intent (Scenarios)
            'Clicked_Link_6Months', 'Clicked_Link_Frequency', 'Link_Clicking_Context',
            'UCC_Scenario_Response', 'UCC_Scenario_Response_Time',
            'Bank_Scenario_Response', 'Bank_Scenario_Response_Time',
            'Social_Media_Scenario_Response', 'Prize_Scenario_Response',
            'Urgent_Action_Scenario_Response', 'Familiar_Sender_Scenario_Response',
            'Scenario_Rationale', 'Confidence_Level', 'Confidence_Calibration',
            'Report_Phishing_Knowledge', 'Report_Phishing_Willingness',
            'Behavior_Score', 'Behavior_Score_Percentage',
            
            # Section E: Security Behaviors & Experiences
            'Entered_Credentials_After_Link', 'Credential_Entry_Context',
            'Financial_Loss_Experienced', 'Financial_Loss_Amount',
            'Account_Compromise_Experienced', 'Compromise_Type',
            'Two_Factor_Usage', 'Two_Factor_Adoption_Barriers',
            'Password_Manager_Usage', 'Password_Reuse_Practice',
            'Password_Strength_Awareness', 'Security_Update_Practice',
            'Device_Security_Measures', 'Public_WiFi_Risks_Awareness',
            'Public_WiFi_Behavior', 'Security_Incident_Reporting',
            'Recovery_Steps_Knowledge',
            
            # Section F: Attitudes & Recommendations
            'Tech_Degree_Opinion', 'Tech_Degree_Opinion_Rationale',
            'Cybersecurity_Attitude', 'Perceived_Vulnerability',
            'Perceived_Severity', 'Response_Efficacy', 'Self_Efficacy',
            'Training_Compulsory_Opinion', 'Training_Frequency_Preference',
            'Training_Format_Preference', 'Most_Effective_Method',
            'UCC_Security_Adequacy', 'UCC_Security_Improvement_Suggestions',
            'Willingness_Participate_Future', 'Interest_Advanced_Training',
            
            # Section G: Advanced Metrics (Calculated)
            'ABG_Index', 'ABG_Index_Interpretation', 'Total_Susceptibility_Score',
            'Susceptibility_Level', 'STEM_NonSTEM_Gap_Contribution',
            'Awareness_Score', 'Behavior_Score', 'Attitude_Score',
            'Skipped_Sections', 'Incomplete_Sections', 'Survey_Version'
        ]
        with open(CSV_FILE, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(headers)
        print(f"✓ Created comprehensive CSV file: {CSV_FILE}")

def classify_stem(program, college):
    """Enhanced STEM classification with confidence scoring"""
    if not program or not college:
        return 'Unknown', 0
    
    program_lower = program.lower()
    college_lower = college.lower()
    
    # Comprehensive STEM keywords
    stem_keywords = {
        'computer': 1.0, 'information technology': 1.0, 'computer science': 1.0,
        'software': 1.0, 'hardware': 1.0, 'networking': 1.0, 'cybersecurity': 1.0,
        'engineering': 0.9, 'mechanical': 0.9, 'electrical': 0.9, 'civil': 0.9,
        'chemical': 0.9, 'biomedical': 0.9, 'aerospace': 0.9,
        'mathematics': 0.8, 'statistics': 0.8, 'applied mathematics': 0.8,
        'physics': 0.9, 'chemistry': 0.9, 'biology': 0.9, 'biochemistry': 0.9,
        'microbiology': 0.9, 'molecular biology': 0.9, 'biotechnology': 0.9,
        'agriculture': 0.7, 'agronomy': 0.7, 'crop science': 0.7,
        'medicine': 0.9, 'nursing': 0.8, 'pharmacy': 0.9, 'dentistry': 0.9,
        'public health': 0.7, 'health sciences': 0.8, 'allied health': 0.7,
        'data science': 1.0, 'artificial intelligence': 1.0, 'machine learning': 1.0,
        'robotics': 1.0, 'telecommunications': 0.9, 'electronics': 0.9
    }
    
    stem_colleges = ['agriculture', 'natural sciences', 'health', 'allied sciences', 
                     'physical sciences', 'biological sciences', 'engineering']
    
    confidence = 0
    
    # Check college first
    for college_key in stem_colleges:
        if college_key in college_lower:
            confidence = max(confidence, 0.7)
            break
    
    # Check program with weighted keywords
    for keyword, weight in stem_keywords.items():
        if keyword in program_lower:
            confidence = max(confidence, weight)
    
    if confidence >= 0.5:
        return 'STEM', confidence
    elif confidence > 0:
        return 'Borderline STEM', confidence
    else:
        return 'Non-STEM', confidence

def calculate_advanced_scores(data, skipped_sections):
    """Calculate comprehensive scores"""
    
    # Knowledge Score (0-100%)
    k_score = 0
    
    # Definition
    if data.get('phishing_def') == 'Attempts to trick users into revealing sensitive information':
        k_score += 20
    
    # Indicators - at least 4 correct
    indicators = data.get('indicators', [])
    correct_indicators = {'Urgent language', 'Password requests', 'Suspicious links', 
                         'Poor grammar', 'Unexpected prizes', 'Generic greetings'}
    correct_count = sum(1 for i in indicators if i in correct_indicators)
    k_score += min(25, (correct_count / 6) * 25)
    
    # Password statement
    if data.get('password_stmt') == 'True':
        k_score += 15
    
    # URL Hovering
    if data.get('url_hovering') == 'Yes':
        k_score += 10
    
    # HTTPS knowledge
    if data.get('https_knowledge') == 'Yes':
        k_score += 10
    
    # Padlock icon
    if data.get('padlock_knowledge') == 'Yes':
        k_score += 10
    
    # Spear phishing
    if data.get('spear_phishing') == 'Yes':
        k_score += 10
    
    knowledge_score = min(100, k_score)
    
    # Behavior Score (0-100%)
    b_score = 50  # Start at 50 (neutral)
    
    # Clicked links - negative
    if data.get('clicked_link') == 'Yes':
        b_score -= 20
    elif data.get('clicked_link') == 'No':
        b_score += 10
    
    # UCC scenario
    safe_ucc = ['Log in directly to official portal', 'Forward to IT']
    if data.get('ucc_scenario') in safe_ucc:
        b_score += 15
    elif data.get('ucc_scenario') == 'Click link immediately':
        b_score -= 15
    
    # Bank scenario
    safe_bank = ['Call bank officially', 'Forward to fraud dept']
    if data.get('bank_scenario') in safe_bank:
        b_score += 15
    elif data.get('bank_scenario') == 'Click and provide info':
        b_score -= 15
    
    # 2FA usage
    if data.get('uses_2fa') == 'Always':
        b_score += 20
    elif data.get('uses_2fa') == 'Sometimes':
        b_score += 10
    
    # Password manager
    if data.get('password_manager') == 'Yes':
        b_score += 10
    
    # Reporting intent
    if data.get('report_intent') == 'Yes':
        b_score += 10
    
    behavior_score = max(0, min(100, b_score))
    
    # Awareness-Behavior Gap (ABG)
    abg_index = knowledge_score - behavior_score
    
    # Total Susceptibility Score
    total_susceptibility = (100 - knowledge_score) * 0.3 + (100 - behavior_score) * 0.7
    
    # Interpretation
    if abg_index > 20:
        abg_interpretation = "Overconfident (Awareness exceeds safe behavior)"
    elif abg_index < -20:
        abg_interpretation = "Cautious but uninformed (Behavior better than knowledge)"
    else:
        abg_interpretation = "Balanced (Knowledge aligns with behavior)"
    
    # Susceptibility Level
    if total_susceptibility < 30:
        susceptibility_level = "Low"
    elif total_susceptibility < 50:
        susceptibility_level = "Moderate"
    elif total_susceptibility < 70:
        susceptibility_level = "High"
    else:
        susceptibility_level = "Critical"
    
    return {
        'knowledge_score': knowledge_score,
        'knowledge_percentage': knowledge_score,
        'behavior_score': behavior_score,
        'behavior_percentage': behavior_score,
        'abg_index': abg_index,
        'abg_interpretation': abg_interpretation,
        'total_susceptibility': total_susceptibility,
        'susceptibility_level': susceptibility_level
    }

def save_to_csv(row_data):
    """Save response to CSV file"""
    with open(CSV_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(row_data)
    print(f"✓ Response saved to {CSV_FILE}")

# Initialize CSV file
ensure_csv_headers()

# Global CSS for consistent styling
BASE_STYLES = '''
    * {
        margin: 0;
        padding: 0;
        box-sizing: border-box;
    }
    
    body {
        font-family: 'Poppins', 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        min-height: 100vh;
        padding: 20px;
        position: relative;
        overflow-x: hidden;
    }
    
    /* Animated background */
    body::before {
        content: '';
        position: fixed;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        background: 
            radial-gradient(circle at 20% 50%, rgba(255,255,255,0.1) 0%, transparent 50%),
            radial-gradient(circle at 80% 80%, rgba(255,255,255,0.1) 0%, transparent 50%);
        pointer-events: none;
        z-index: 0;
    }
    
    .container {
        max-width: 900px;
        margin: 0 auto;
        position: relative;
        z-index: 1;
    }
    
    /* Progress Bar */
    .progress-container {
        margin-bottom: 30px;
    }
    
    .progress-stats {
        display: flex;
        justify-content: space-between;
        color: white;
        margin-bottom: 10px;
        font-size: 0.9rem;
        text-shadow: 0 2px 4px rgba(0,0,0,0.2);
    }
    
    .progress-bar {
        background: rgba(255,255,255,0.2);
        height: 12px;
        border-radius: 20px;
        overflow: hidden;
        backdrop-filter: blur(5px);
        border: 1px solid rgba(255,255,255,0.3);
    }
    
    .progress-fill {
        background: linear-gradient(90deg, #ffd700, #ffa500);
        height: 100%;
        border-radius: 20px;
        transition: width 0.5s ease;
        position: relative;
        overflow: hidden;
    }
    
    .progress-fill::after {
        content: '';
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        background: linear-gradient(90deg, transparent, rgba(255,255,255,0.3), transparent);
        animation: shimmer 2s infinite;
    }
    
    @keyframes shimmer {
        0% { transform: translateX(-100%); }
        100% { transform: translateX(100%); }
    }
    
    /* Card Styles */
    .card {
        background: rgba(255, 255, 255, 0.95);
        backdrop-filter: blur(10px);
        border-radius: 30px;
        overflow: hidden;
        box-shadow: 0 30px 70px rgba(0,0,0,0.3);
        animation: slideUp 0.6s cubic-bezier(0.23, 1, 0.32, 1);
        border: 1px solid rgba(255,255,255,0.2);
    }
    
    @keyframes slideUp {
        from {
            opacity: 0;
            transform: translateY(50px);
        }
        to {
            opacity: 1;
            transform: translateY(0);
        }
    }
    
    /* Header Styles */
    .card-header {
        background: linear-gradient(135deg, #0c2461 0%, #1e3799 100%);
        padding: 40px;
        color: white;
        text-align: center;
        position: relative;
        overflow: hidden;
    }
    
    .card-header::before {
        content: '';
        position: absolute;
        top: -50%;
        right: -50%;
        width: 200%;
        height: 200%;
        background: radial-gradient(circle, rgba(255,255,255,0.1) 0%, transparent 70%);
        animation: rotate 20s linear infinite;
    }
    
    @keyframes rotate {
        from { transform: rotate(0deg); }
        to { transform: rotate(360deg); }
    }
    
    .card-header h1 {
        font-size: 2.2rem;
        margin-bottom: 10px;
        position: relative;
        text-shadow: 2px 2px 4px rgba(0,0,0,0.2);
    }
    
    .card-header h2 {
        font-size: 1.3rem;
        font-weight: normal;
        opacity: 0.9;
        position: relative;
    }
    
    .badge {
        display: inline-block;
        background: rgba(255,215,0,0.2);
        color: #ffd700;
        padding: 8px 25px;
        border-radius: 50px;
        margin-top: 15px;
        font-weight: bold;
        border: 2px solid #ffd700;
        position: relative;
        backdrop-filter: blur(5px);
    }
    
    /* Content */
    .card-content {
        padding: 40px;
    }
    
    /* Form Elements */
    .form-group {
        margin-bottom: 30px;
        animation: fadeIn 0.5s ease-out;
    }
    
    @keyframes fadeIn {
        from {
            opacity: 0;
            transform: translateX(-20px);
        }
        to {
            opacity: 1;
            transform: translateX(0);
        }
    }
    
    label {
        display: block;
        margin-bottom: 15px;
        font-weight: 600;
        color: #2c3e50;
        font-size: 1.1rem;
    }
    
    .label-highlight {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 5px 15px;
        border-radius: 50px;
        display: inline-block;
        margin-right: 10px;
        font-size: 0.9rem;
    }
    
    select, input[type="text"], textarea {
        width: 100%;
        padding: 15px 20px;
        border: 2px solid #e0e0e0;
        border-radius: 15px;
        font-size: 16px;
        transition: all 0.3s;
        background: white;
        box-shadow: 0 4px 6px rgba(0,0,0,0.05);
    }
    
    select:focus, input:focus, textarea:focus {
        outline: none;
        border-color: #667eea;
        box-shadow: 0 10px 20px rgba(102, 126, 234, 0.2);
        transform: translateY(-2px);
    }
    
    /* Radio & Checkbox Styles */
    .option-group {
        background: #f8fafc;
        border-radius: 20px;
        padding: 20px;
        border: 2px solid transparent;
        transition: all 0.3s;
    }
    
    .option-group:hover {
        border-color: #667eea;
        background: white;
        box-shadow: 0 10px 30px rgba(0,0,0,0.1);
    }
    
    .option-item {
        margin: 15px 0;
        padding: 12px;
        background: white;
        border-radius: 12px;
        transition: all 0.3s;
        cursor: pointer;
        border: 2px solid transparent;
    }
    
    .option-item:hover {
        transform: translateX(10px);
        border-color: #667eea;
        box-shadow: 0 5px 15px rgba(102, 126, 234, 0.2);
    }
    
    .option-item input[type="radio"],
    .option-item input[type="checkbox"] {
        margin-right: 15px;
        width: 20px;
        height: 20px;
        cursor: pointer;
        accent-color: #667eea;
    }
    
    .option-item label {
        display: inline;
        margin: 0;
        font-weight: normal;
        cursor: pointer;
    }
    
    /* Scenario Box */
    .scenario-box {
        background: linear-gradient(135deg, #f6f9fc 0%, #e6f0f9 100%);
        border-radius: 20px;
        padding: 25px;
        margin: 20px 0;
        border-left: 5px solid #667eea;
        position: relative;
        overflow: hidden;
    }
    
    .scenario-box::before {
        content: '🔍';
        position: absolute;
        top: 10px;
        right: 10px;
        font-size: 2rem;
        opacity: 0.2;
    }
    
    .scenario-box h4 {
        color: #0c2461;
        margin-bottom: 15px;
        font-size: 1.2rem;
    }
    
    .scenario-text {
        background: white;
        padding: 20px;
        border-radius: 15px;
        margin-bottom: 20px;
        font-style: italic;
        color: #2c3e50;
        box-shadow: 0 4px 6px rgba(0,0,0,0.05);
        border: 1px solid #e0e0e0;
    }
    
    /* Buttons */
    .btn {
        padding: 15px 30px;
        border: none;
        border-radius: 50px;
        font-size: 1.1rem;
        font-weight: bold;
        cursor: pointer;
        transition: all 0.3s;
        margin: 10px;
        text-decoration: none;
        display: inline-block;
        box-shadow: 0 4px 6px rgba(0,0,0,0.1);
    }
    
    .btn-primary {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
    }
    
    .btn-primary:hover {
        transform: translateY(-3px);
        box-shadow: 0 10px 30px rgba(102, 126, 234, 0.4);
    }
    
    .btn-success {
        background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%);
        color: white;
    }
    
    .btn-success:hover {
        transform: translateY(-3px);
        box-shadow: 0 10px 30px rgba(17, 153, 142, 0.4);
    }
    
    .btn-large {
        padding: 18px 40px;
        font-size: 1.2rem;
        width: 100%;
        max-width: 400px;
        margin: 20px auto;
        display: block;
    }
    
    .button-group {
        display: flex;
        gap: 15px;
        margin-top: 30px;
        flex-wrap: wrap;
    }
    
    /* Info Cards */
    .info-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
        gap: 25px;
        margin: 30px 0;
    }
    
    .info-card {
        background: #f8fafc;
        padding: 30px;
        border-radius: 20px;
        text-align: center;
        transition: all 0.3s;
        border: 2px solid transparent;
    }
    
    .info-card:hover {
        transform: translateY(-10px);
        border-color: #667eea;
        box-shadow: 0 20px 40px rgba(0,0,0,0.1);
    }
    
    .info-icon {
        font-size: 3rem;
        margin-bottom: 15px;
    }
    
    /* Team Section */
    .team-section {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 40px;
        border-radius: 30px;
        margin: 30px 0;
        text-align: center;
    }
    
    .team-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
        gap: 20px;
        margin: 30px 0;
    }
    
    .team-member {
        background: rgba(255,255,255,0.1);
        padding: 20px;
        border-radius: 20px;
        backdrop-filter: blur(5px);
        transition: all 0.3s;
    }
    
    .team-member:hover {
        transform: translateY(-5px);
        background: rgba(255,255,255,0.2);
    }
    
    .member-name {
        font-size: 1.2rem;
        font-weight: bold;
        margin-bottom: 5px;
    }
    
    .member-role {
        font-size: 0.9rem;
        opacity: 0.9;
    }
    
    .supervisor {
        margin-top: 20px;
        padding: 20px;
        border-top: 2px solid rgba(255,255,255,0.2);
    }
    
    .supervisor-name {
        font-size: 1.3rem;
        color: #ffd700;
        font-weight: bold;
    }
    
    /* Stats Preview */
    .stats-preview {
        display: grid;
        grid-template-columns: repeat(3, 1fr);
        gap: 15px;
        background: linear-gradient(135deg, #f6f9fc 0%, #e6f0f9 100%);
        padding: 25px;
        border-radius: 20px;
        margin: 30px 0;
    }
    
    .stat-item {
        text-align: center;
    }
    
    .stat-value {
        font-size: 2rem;
        font-weight: bold;
        color: #667eea;
    }
    
    .stat-label {
        color: #2c3e50;
        font-size: 0.9rem;
        margin-top: 5px;
    }
    
    /* Footer */
    .footer {
        text-align: center;
        margin-top: 30px;
        color: rgba(255,255,255,0.9);
        font-size: 0.9rem;
    }
    
    /* Responsive */
    @media (max-width: 768px) {
        .card-header h1 { font-size: 1.8rem; }
        .card-content { padding: 20px; }
        .stats-preview { grid-template-columns: 1fr; }
        .button-group { flex-direction: column; }
    }
'''

# Home Page
@app.route('/')
def index():
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>UCC Cybersecurity Research Study</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>{BASE_STYLES}</style>
    </head>
    <body>
        <div class="container">
            <div class="card">
                <div class="card-header">
                    <h1>UNIVERSITY OF CAPE COAST</h1>
                    <h2>Department of Information Technology</h2>
                    <div class="badge">📊 IRB Approved Study: UCC/IT/2026/PHISH/001</div>
                </div>
                
                <div class="card-content">
                    <div class="stats-preview">
                        <div class="stat-item">
                            <div class="stat-value">250+</div>
                            <div class="stat-label">Target Participants</div>
                        </div>
                        <div class="stat-item">
                            <div class="stat-value">10-12</div>
                            <div class="stat-label">Minutes</div>
                        </div>
                        <div class="stat-item">
                            <div class="stat-value">🔒</div>
                            <div class="stat-label">Anonymous</div>
                        </div>
                    </div>
                    
                    <div class="info-grid">
                        <div class="info-card">
                            <div class="info-icon">🎓</div>
                            <h3>Final Year Project</h3>
                            <p>BSc Information Technology group research investigating cybersecurity awareness across academic disciplines</p>
                        </div>
                        <div class="info-card">
                            <div class="info-icon">🔬</div>
                            <h3>Research Focus</h3>
                            <p>Comparative analysis of phishing susceptibility between STEM and Non-STEM students</p>
                        </div>
                        <div class="info-card">
                            <div class="info-icon">📈</div>
                            <h3>Impact</h3>
                            <p>Your participation will help improve UCC's cybersecurity education programs</p>
                        </div>
                    </div>
                    
                    <div class="team-section">
                        <h2>👥 Research Team</h2>
                        <div class="team-grid" style="grid-template-columns: 1fr;">
                            <div class="team-member">
                                <div class="member-name">BSc Information Technology</div>
                                <div class="member-role">Final Year Group Project</div>
                            </div>
                        </div>
                        <div class="supervisor">
                            <p>Supervisor</p>
                            <div class="supervisor-name">Dr. Franklin Amoh, PhD</div>
                        </div>
                    </div>
                    
                    <form action="/start" method="POST" style="text-align: center;">
                        <button class="btn btn-primary btn-large" type="submit">
                            🚀 Begin Research Study
                        </button>
                    </form>
                    
                    <div style="text-align: center; margin-top: 20px; color: #666;">
                        <p>For questions: <a href="mailto:it.department@ucc.edu.gh" style="color: #667eea;">it.department@ucc.edu.gh</a></p>
                    </div>
                </div>
            </div>
            
            <div class="footer">
                <p>© 2026 University of Cape Coast | School of Information & Communication Studies</p>
            </div>
        </div>
    </body>
    </html>
    '''

@app.route('/start', methods=['POST'])
def start():
    session.clear()
    session['response_id'] = datetime.now().strftime('UCC%Y%m%d%H%M%S') + '-' + os.urandom(4).hex()
    session['start_time'] = datetime.now().isoformat()
    session['skipped_sections'] = []
    session['page_sequence'] = []
    session['response_times'] = {}
    return redirect('/consent')

@app.route('/consent', methods=['GET', 'POST'])
def consent():
    if request.method == 'POST':
        session['page_sequence'].append('consent')
        session['response_times']['consent'] = datetime.now().isoformat()
        return redirect('/demographics')
    
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Informed Consent - UCC Research</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>{BASE_STYLES}</style>
    </head>
    <body>
        <div class="container">
            <div class="progress-container">
                <div class="progress-stats">
                    <span>Step 1 of 6</span>
                    <span>Consent Form</span>
                </div>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: 10%"></div>
                </div>
            </div>
            
            <div class="card">
                <div class="card-header">
                    <h1>INFORMED CONSENT FORM</h1>
                    <p>Department of Information Technology</p>
                </div>
                
                <div class="card-content">
                    <div style="background: #f8fafc; border-radius: 20px; padding: 30px; margin-bottom: 30px;">
                        <h3 style="color: #0c2461; margin-bottom: 15px;">📋 Study Overview</h3>
                        <p><strong>Title:</strong> A Comparative Analysis of Phishing Susceptibility between STEM and Non-STEM Students at UCC</p>
                        <p><strong>Research Team:</strong> BSc Information Technology Final Year Group Project</p>
                        <p><strong>Supervisor:</strong> Dr. Franklin Amoh, PhD</p>
                    </div>
                    
                    <div class="info-grid">
                        <div class="info-card">
                            <h4>🎯 Purpose</h4>
                            <p>Assess phishing awareness and compare susceptibility between STEM and Non-STEM students</p>
                        </div>
                        <div class="info-card">
                            <h4>📝 Participation</h4>
                            <p>10-12 minute anonymous questionnaire across 6 sections</p>
                        </div>
                        <div class="info-card">
                            <h4>🔒 Confidentiality</h4>
                            <p>No personal data collected. All responses encrypted and secure</p>
                        </div>
                        <div class="info-card">
                            <h4>✅ Voluntary</h4>
                            <p>You may withdraw at any time without penalty</p>
                        </div>
                    </div>
                    
                    <form method="POST" onsubmit="return validateConsent()">
                        <div style="background: #f0f8ff; border-radius: 20px; padding: 30px; margin: 30px 0;">
                            <h3 style="color: #0c2461; margin-bottom: 20px;">Consent Confirmation</h3>
                            
                            <div class="option-item">
                                <input type="checkbox" id="understand" required>
                                <label for="understand">I have read and understood the study information</label>
                            </div>
                            
                            <div class="option-item">
                                <input type="checkbox" id="voluntary" required>
                                <label for="voluntary">I understand that participation is voluntary</label>
                            </div>
                            
                            <div class="option-item">
                                <input type="checkbox" id="anonymous" required>
                                <label for="anonymous">I understand that responses are anonymous</label>
                            </div>
                            
                            <div class="option-item">
                                <input type="checkbox" id="data_use" required>
                                <label for="data_use">I consent to my anonymized data being used for research</label>
                            </div>
                            
                            <div class="option-item">
                                <input type="checkbox" id="agree_all" required>
                                <label for="agree_all"><strong>I VOLUNTARILY AGREE TO PARTICIPATE</strong></label>
                            </div>
                        </div>
                        
                        <div class="button-group">
                            <button type="submit" class="btn btn-primary" style="flex: 2;">✓ I CONSENT - BEGIN SURVEY</button>
                            <a href="/" class="btn" style="background: #6c757d; color: white; flex: 1;">✗ DECLINE</a>
                        </div>
                    </form>
                    
                    <div style="text-align: center; margin-top: 30px; color: #666;">
                        <p>Contact: IT Department - it.department@ucc.edu.gh</p>
                        <p>Supervisor: Dr. Franklin Amoh - famoh@ucc.edu.gh</p>
                    </div>
                </div>
            </div>
        </div>
        
        <script>
            function validateConsent() {{
                const checkboxes = document.querySelectorAll('input[type="checkbox"]');
                let allChecked = true;
                checkboxes.forEach(cb => {{ if (!cb.checked) allChecked = false; }});
                if (!allChecked) {{
                    alert('Please confirm all consent statements to proceed');
                    return false;
                }}
                return true;
            }}
        </script>
    </body>
    </html>
    '''

# Demographics Section
@app.route('/demographics', methods=['GET', 'POST'])
def demographics():
    if request.method == 'POST':
        session['demo'] = {
            'gender': request.form.get('gender'),
            'age': request.form.get('age'),
            'level': request.form.get('level'),
            'college': request.form.get('college'),
            'program': request.form.get('program'),
            'stem_confidence': request.form.get('stem_confidence')
        }
        session['page_sequence'].append('demographics')
        session['response_times']['demographics'] = datetime.now().isoformat()
        return redirect('/digital')
    
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Demographics - UCC Research</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>{BASE_STYLES}</style>
    </head>
    <body>
        <div class="container">
            <div class="progress-container">
                <div class="progress-stats">
                    <span>Step 2 of 6</span>
                    <span>Section A: Demographics</span>
                </div>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: 20%"></div>
                </div>
            </div>
            
            <div class="card">
                <div class="card-header">
                    <h1>📋 Section A: Demographics</h1>
                    <p>Help us understand the diverse backgrounds of UCC students</p>
                </div>
                
                <div class="card-content">
                    <form method="POST">
                        <div class="form-group">
                            <label><span class="label-highlight">1</span> Gender *</label>
                            <div class="option-group">
                                <div class="option-item"><input type="radio" name="gender" value="Male" required> Male</div>
                                <div class="option-item"><input type="radio" name="gender" value="Female"> Female</div>
                                <div class="option-item"><input type="radio" name="gender" value="Prefer not to say"> Prefer not to say</div>
                            </div>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">2</span> Age Range *</label>
                            <select name="age" required>
                                <option value="">-- Select age range --</option>
                                <option value="Below 18">Below 18 years</option>
                                <option value="18-21">18-21 years</option>
                                <option value="22-25">22-25 years</option>
                                <option value="26-30">26-30 years</option>
                                <option value="Above 30">Above 30 years</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">3</span> Level of Study *</label>
                            <select name="level" required>
                                <option value="">-- Select level --</option>
                                <option value="Level 100">Level 100</option>
                                <option value="Level 200">Level 200</option>
                                <option value="Level 300">Level 300</option>
                                <option value="Level 400">Level 400</option>
                                <option value="Postgraduate">Postgraduate</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">4</span> College/School *</label>
                            <select name="college" required>
                                <option value="">-- Select college --</option>
                                <option value="Agriculture and Natural Sciences">College of Agriculture and Natural Sciences</option>
                                <option value="Humanities and Legal Studies">College of Humanities and Legal Studies</option>
                                <option value="Education Studies">College of Education Studies</option>
                                <option value="Health and Allied Sciences">College of Health and Allied Sciences</option>
                                <option value="School of Business">School of Business</option>
                                <option value="Other">Other (please specify)</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">5</span> Program of Study *</label>
                            <input type="text" name="program" required placeholder="e.g., BSc Information Technology, BEd Mathematics, BA Economics">
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">6</span> Technology Confidence *</label>
                            <select name="stem_confidence" required>
                                <option value="">-- Select confidence level --</option>
                                <option value="Very Low">Very Low (I struggle with basic tasks)</option>
                                <option value="Low">Low (I can do basic tasks)</option>
                                <option value="Moderate">Moderate (Comfortable with common tasks)</option>
                                <option value="High">High (I can troubleshoot most issues)</option>
                                <option value="Very High">Very High (I'm tech-savvy/expert)</option>
                            </select>
                        </div>
                        
                        <button class="btn btn-primary btn-large" type="submit">Continue to Section B →</button>
                    </form>
                </div>
            </div>
        </div>
    </body>
    </html>
    '''

# Digital Usage Section
@app.route('/digital', methods=['GET', 'POST'])
def digital():
    if request.method == 'POST':
        session['digital'] = {
            'email_freq': request.form.get('email_freq'),
            'email_count': request.form.get('email_count'),
            'email_provider': request.form.get('email_provider'),
            'banking_freq': request.form.get('banking_freq'),
            'mobile_money': request.form.get('mobile_money'),
            'shopping_freq': request.form.get('shopping_freq'),
            'social_count': request.form.get('social_count'),
            'primary_social': request.form.get('primary_social'),
            'internet_years': request.form.get('internet_years'),
            'suspicious_msgs': request.form.get('suspicious_msgs'),
            'suspicious_freq': request.form.get('suspicious_freq'),
            'suspicious_channels': request.form.get('suspicious_channels'),
            'training_received': request.form.get('training_received'),
            'training_types': request.form.get('training_types'),
            'digital_literacy': request.form.get('digital_literacy')
        }
        session['page_sequence'].append('digital')
        session['response_times']['digital'] = datetime.now().isoformat()
        return redirect('/knowledge')
    
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Digital Usage - UCC Research</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>{BASE_STYLES}</style>
    </head>
    <body>
        <div class="container">
            <div class="progress-container">
                <div class="progress-stats">
                    <span>Step 3 of 6</span>
                    <span>Section B: Digital Usage</span>
                </div>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: 35%"></div>
                </div>
            </div>
            
            <div class="card">
                <div class="card-header">
                    <h1>💻 Section B: Digital Usage Patterns</h1>
                    <p>Tell us about your online habits and digital experience</p>
                </div>
                
                <div class="card-content">
                    <form method="POST">
                        <div class="form-group">
                            <label><span class="label-highlight">7</span> How often do you use email? *</label>
                            <select name="email_freq" required>
                                <option value="">-- Select frequency --</option>
                                <option value="Daily">Daily</option>
                                <option value="Several times a week">Several times a week</option>
                                <option value="Weekly">Weekly</option>
                                <option value="Occasionally">Occasionally</option>
                                <option value="Rarely">Rarely</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">8</span> How many email accounts do you have? *</label>
                            <select name="email_count" required>
                                <option value="">-- Select --</option>
                                <option value="1">1 account</option>
                                <option value="2-3">2-3 accounts</option>
                                <option value="4-5">4-5 accounts</option>
                                <option value="6+">6+ accounts</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">9</span> Primary email provider? *</label>
                            <select name="email_provider" required>
                                <option value="">-- Select --</option>
                                <option value="Gmail">Gmail</option>
                                <option value="Yahoo">Yahoo</option>
                                <option value="Outlook">Outlook/Hotmail</option>
                                <option value="UCC Mail">UCC Student Mail</option>
                                <option value="Other">Other</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">10</span> Mobile banking frequency? *</label>
                            <select name="banking_freq" required>
                                <option value="">-- Select --</option>
                                <option value="Daily">Daily</option>
                                <option value="Weekly">Weekly</option>
                                <option value="Monthly">Monthly</option>
                                <option value="Rarely">Rarely</option>
                                <option value="Never">Never</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">11</span> Do you use Mobile Money? *</label>
                            <div class="option-group">
                                <div class="option-item"><input type="radio" name="mobile_money" value="Yes, frequently" required> Yes, frequently</div>
                                <div class="option-item"><input type="radio" name="mobile_money" value="Yes, occasionally"> Yes, occasionally</div>
                                <div class="option-item"><input type="radio" name="mobile_money" value="No"> No</div>
                            </div>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">12</span> Online shopping frequency? *</label>
                            <select name="shopping_freq" required>
                                <option value="">-- Select --</option>
                                <option value="Weekly">Weekly</option>
                                <option value="Monthly">Monthly</option>
                                <option value="Few times a year">Few times a year</option>
                                <option value="Rarely">Rarely</option>
                                <option value="Never">Never</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">13</span> Social media platforms used? *</label>
                            <select name="social_count" required>
                                <option value="">-- Select --</option>
                                <option value="1-2">1-2 platforms</option>
                                <option value="3-4">3-4 platforms</option>
                                <option value="5+">5+ platforms</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">14</span> Years of internet experience? *</label>
                            <select name="internet_years" required>
                                <option value="">-- Select --</option>
                                <option value="<1">Less than 1 year</option>
                                <option value="1-3">1-3 years</option>
                                <option value="4-6">4-6 years</option>
                                <option value="7-10">7-10 years</option>
                                <option value="10+">10+ years</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">15</span> Received suspicious messages? *</label>
                            <select name="suspicious_msgs" required>
                                <option value="">-- Select --</option>
                                <option value="Yes, frequently">Yes, frequently</option>
                                <option value="Yes, occasionally">Yes, occasionally</option>
                                <option value="Yes, rarely">Yes, rarely</option>
                                <option value="No">No</option>
                                <option value="Not sure">Not sure</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">16</span> Received cybersecurity training? *</label>
                            <div class="option-group">
                                <div class="option-item"><input type="radio" name="training_received" value="Yes, formal" required> Yes, formal training</div>
                                <div class="option-item"><input type="radio" name="training_received" value="Yes, informal"> Yes, informal (online, friends)</div>
                                <div class="option-item"><input type="radio" name="training_received" value="No"> No training</div>
                            </div>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">17</span> Self-rated digital literacy? *</label>
                            <select name="digital_literacy" required>
                                <option value="">-- Select --</option>
                                <option value="Beginner">Beginner - I need help often</option>
                                <option value="Intermediate">Intermediate - I can manage</option>
                                <option value="Advanced">Advanced - I'm comfortable</option>
                                <option value="Expert">Expert - I can teach others</option>
                            </select>
                        </div>
                        
                        <button class="btn btn-primary btn-large" type="submit">Continue to Section C →</button>
                    </form>
                </div>
            </div>
        </div>
    </body>
    </html>
    '''

# Knowledge Section
@app.route('/knowledge', methods=['GET', 'POST'])
def knowledge():
    if request.method == 'POST':
        familiar = request.form.get('familiar')
        
        if familiar == 'No':
            session['knowledge'] = {
                'familiar': 'No',
                'phishing_def': 'Not asked (skipped)',
                'indicators': ['Not asked (skipped)'],
                'password_stmt': 'Not asked (skipped)',
                'url_hovering': 'Not asked (skipped)',
                'https_knowledge': 'Not asked (skipped)',
                'padlock_knowledge': 'Not asked (skipped)',
                'spear_phishing': 'Not asked (skipped)'
            }
            session['skipped_sections'].extend(['knowledge', 'behavior', 'security'])
            return redirect('/attitudes')
        else:
            indicators = request.form.getlist('indicators')
            session['knowledge'] = {
                'familiar': 'Yes',
                'phishing_def': request.form.get('definition'),
                'indicators': indicators,
                'password_stmt': request.form.get('password_stmt'),
                'url_hovering': request.form.get('url_hovering'),
                'https_knowledge': request.form.get('https_knowledge'),
                'padlock_knowledge': request.form.get('padlock_knowledge'),
                'spear_phishing': request.form.get('spear_phishing')
            }
            return redirect('/behavior')
    
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Phishing Knowledge - UCC Research</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>{BASE_STYLES}</style>
        <style>
            .knowledge-tip {{
                background: linear-gradient(135deg, #fff3cd 0%, #ffeaa7 100%);
                padding: 15px;
                border-radius: 15px;
                margin: 20px 0;
                border-left: 5px solid #ffa500;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="progress-container">
                <div class="progress-stats">
                    <span>Step 4 of 6</span>
                    <span>Section C: Phishing Knowledge</span>
                </div>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: 50%"></div>
                </div>
            </div>
            
            <div class="card">
                <div class="card-header">
                    <h1>🎣 Section C: Phishing Knowledge</h1>
                    <p>Test your understanding of phishing threats</p>
                </div>
                
                <div class="card-content">
                    <div class="knowledge-tip">
                        <strong>💡 Note:</strong> If you're not familiar with phishing, you can skip this section. Honest answers help our research!
                    </div>
                    
                    <form method="POST">
                        <div class="form-group">
                            <label><span class="label-highlight">18</span> Are you familiar with the term "phishing"? *</label>
                            <div class="option-group" id="familiarGroup">
                                <div class="option-item"><input type="radio" name="familiar" value="Yes" required onclick="toggleQuestions(true)"> Yes, I'm familiar</div>
                                <div class="option-item"><input type="radio" name="familiar" value="No" required onclick="toggleQuestions(false)"> No, not familiar (skip technical questions)</div>
                            </div>
                        </div>
                        
                        <div id="knowledgeQuestions" style="display: none;">
                            <div class="form-group">
                                <label><span class="label-highlight">19</span> Phishing primarily refers to: *</label>
                                <div class="option-group">
                                    <div class="option-item"><input type="radio" name="definition" value="Attempts to trick users into revealing sensitive information"> Attempts to trick users into revealing sensitive information</div>
                                    <div class="option-item"><input type="radio" name="definition" value="A type of antivirus software"> A type of antivirus software</div>
                                    <div class="option-item"><input type="radio" name="definition" value="A secure authentication method"> A secure authentication method</div>
                                    <div class="option-item"><input type="radio" name="definition" value="Not sure"> Not sure</div>
                                </div>
                            </div>
                            
                            <div class="form-group">
                                <label><span class="label-highlight">20</span> Common phishing indicators (select all): *</label>
                                <div class="option-group">
                                    <div class="option-item"><input type="checkbox" name="indicators" value="Urgent language"> Urgent or threatening language</div>
                                    <div class="option-item"><input type="checkbox" name="indicators" value="Password requests"> Requests for passwords or PINs</div>
                                    <div class="option-item"><input type="checkbox" name="indicators" value="Suspicious links"> Suspicious links or attachments</div>
                                    <div class="option-item"><input type="checkbox" name="indicators" value="Poor grammar"> Poor spelling or grammar</div>
                                    <div class="option-item"><input type="checkbox" name="indicators" value="Unexpected prizes"> Unexpected prize notifications</div>
                                    <div class="option-item"><input type="checkbox" name="indicators" value="Generic greetings"> Generic greetings ("Dear Customer")</div>
                                </div>
                            </div>
                            
                            <div class="form-group">
                                <label><span class="label-highlight">21</span> "Legitimate organizations never ask for passwords via email" - This is: *</label>
                                <div class="option-group">
                                    <div class="option-item"><input type="radio" name="password_stmt" value="True"> True</div>
                                    <div class="option-item"><input type="radio" name="password_stmt" value="False"> False</div>
                                    <div class="option-item"><input type="radio" name="password_stmt" value="Not sure"> Not sure</div>
                                </div>
                            </div>
                            
                            <div class="form-group">
                                <label><span class="label-highlight">22</span> Do you know what "hovering over links" means for security? *</label>
                                <div class="option-group">
                                    <div class="option-item"><input type="radio" name="url_hovering" value="Yes"> Yes</div>
                                    <div class="option-item"><input type="radio" name="url_hovering" value="No"> No</div>
                                    <div class="option-item"><input type="radio" name="url_hovering" value="Not sure"> Not sure</div>
                                </div>
                            </div>
                            
                            <div class="form-group">
                                <label><span class="label-highlight">23</span> What does "HTTPS" indicate? *</label>
                                <div class="option-group">
                                    <div class="option-item"><input type="radio" name="https_knowledge" value="Secure connection"> Secure/encrypted connection</div>
                                    <div class="option-item"><input type="radio" name="https_knowledge" value="Fast website"> Fast website</div>
                                    <div class="option-item"><input type="radio" name="https_knowledge" value="Legitimate website"> Legitimate website</div>
                                    <div class="option-item"><input type="radio" name="https_knowledge" value="Not sure"> Not sure</div>
                                </div>
                            </div>
                            
                            <div class="form-group">
                                <label><span class="label-highlight">24</span> The padlock icon in browser address bar indicates: *</label>
                                <div class="option-group">
                                    <div class="option-item"><input type="radio" name="padlock_knowledge" value="Secure connection"> Secure connection</div>
                                    <div class="option-item"><input type="radio" name="padlock_knowledge" value="Website is safe"> Website is completely safe</div>
                                    <div class="option-item"><input type="radio" name="padlock_knowledge" value="Not sure"> Not sure</div>
                                </div>
                            </div>
                        </div>
                        
                        <button class="btn btn-primary btn-large" type="submit">Continue →</button>
                    </form>
                </div>
            </div>
        </div>
        
        <script>
            function toggleQuestions(show) {{
                document.getElementById('knowledgeQuestions').style.display = show ? 'block' : 'none';
            }}
            
            // Initialize - hide questions initially
            document.addEventListener('DOMContentLoaded', function() {{
                toggleQuestions(false);
            }});
        </script>
    </body>
    </html>
    '''

# Behavior Section
@app.route('/behavior', methods=['GET', 'POST'])
def behavior():
    if 'behavior' in session.get('skipped_sections', []):
        return redirect('/attitudes')
    
    if request.method == 'POST':
        session['behavior'] = {
            'clicked_link': request.form.get('clicked_link'),
            'clicked_freq': request.form.get('clicked_freq'),
            'clicked_context': request.form.get('clicked_context'),
            'ucc_scenario': request.form.get('ucc_scenario'),
            'bank_scenario': request.form.get('bank_scenario'),
            'social_scenario': request.form.get('social_scenario'),
            'prize_scenario': request.form.get('prize_scenario'),
            'urgent_scenario': request.form.get('urgent_scenario'),
            'familiar_scenario': request.form.get('familiar_scenario'),
            'scenario_rationale': request.form.get('scenario_rationale'),
            'confidence': request.form.get('confidence'),
            'report_knowledge': request.form.get('report_knowledge'),
            'report_willingness': request.form.get('report_willingness')
        }
        return redirect('/security')
    
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Behavior Scenarios - UCC Research</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>{BASE_STYLES}</style>
    </head>
    <body>
        <div class="container">
            <div class="progress-container">
                <div class="progress-stats">
                    <span>Step 5 of 6</span>
                    <span>Section D: Behavior Scenarios</span>
                </div>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: 70%"></div>
                </div>
            </div>
            
            <div class="card">
                <div class="card-header">
                    <h1>🧠 Section D: Real-World Scenarios</h1>
                    <p>How would you react in these situations?</p>
                </div>
                
                <div class="card-content">
                    <form method="POST">
                        <div class="form-group">
                            <label><span class="label-highlight">25</span> In the past 6 months, have you clicked a suspicious link? *</label>
                            <select name="clicked_link" required>
                                <option value="">-- Select --</option>
                                <option value="Yes">Yes</option>
                                <option value="No">No</option>
                                <option value="Not sure">Not sure</option>
                            </select>
                        </div>
                        
                        <div class="scenario-box">
                            <h4>📱 Scenario 1: UCC Portal</h4>
                            <div class="scenario-text">
                                "Your student portal will be deactivated in 24hrs. Click here to verify: bit.ly/ucc-portal-verify"
                            </div>
                            <p><strong>Your action? *</strong></p>
                            <div class="option-group">
                                <div class="option-item"><input type="radio" name="ucc_scenario" value="Click link immediately" required> Click link immediately</div>
                                <div class="option-item"><input type="radio" name="ucc_scenario" value="Log in directly to official portal"> Log in directly via official website</div>
                                <div class="option-item"><input type="radio" name="ucc_scenario" value="Ignore and delete"> Ignore and delete</div>
                                <div class="option-item"><input type="radio" name="ucc_scenario" value="Forward to IT"> Forward to IT support</div>
                            </div>
                        </div>
                        
                        <div class="scenario-box">
                            <h4>🏦 Scenario 2: Bank Email</h4>
                            <div class="scenario-text">
                                "50% loan interest reduction! Confirm details now." (Email looks official but has minor spelling errors)
                            </div>
                            <p><strong>Your action? *</strong></p>
                            <div class="option-group">
                                <div class="option-item"><input type="radio" name="bank_scenario" value="Click and provide info" required> Click and provide info</div>
                                <div class="option-item"><input type="radio" name="bank_scenario" value="Call bank officially"> Call bank using official number</div>
                                <div class="option-item"><input type="radio" name="bank_scenario" value="Delete email"> Delete immediately</div>
                                <div class="option-item"><input type="radio" name="bank_scenario" value="Forward to fraud dept"> Forward to bank's fraud dept</div>
                            </div>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">26</span> Confidence in identifying phishing? *</label>
                            <select name="confidence" required>
                                <option value="">-- Select --</option>
                                <option value="Very confident">Very confident</option>
                                <option value="Confident">Confident</option>
                                <option value="Somewhat confident">Somewhat confident</option>
                                <option value="Not confident">Not confident</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">27</span> Would you report phishing attempts? *</label>
                            <select name="report_willingness" required>
                                <option value="">-- Select --</option>
                                <option value="Yes, definitely">Yes, definitely</option>
                                <option value="Probably">Probably</option>
                                <option value="Not sure">Not sure</option>
                                <option value="Probably not">Probably not</option>
                                <option value="No">No</option>
                            </select>
                        </div>
                        
                        <button class="btn btn-primary btn-large" type="submit">Continue to Section E →</button>
                    </form>
                </div>
            </div>
        </div>
    </body>
    </html>
    '''

# Security Section
@app.route('/security', methods=['GET', 'POST'])
def security():
    if 'security' in session.get('skipped_sections', []):
        return redirect('/attitudes')
    
    if request.method == 'POST':
        session['security'] = {
            'entered_creds': request.form.get('entered_creds'),
            'cred_context': request.form.get('cred_context'),
            'financial_loss': request.form.get('financial_loss'),
            'loss_amount': request.form.get('loss_amount'),
            'compromise': request.form.get('compromise'),
            'compromise_type': request.form.get('compromise_type'),
            'two_factor': request.form.get('two_factor'),
            'two_factor_barriers': request.form.get('two_factor_barriers'),
            'password_manager': request.form.get('password_manager'),
            'password_reuse': request.form.get('password_reuse'),
            'password_strength': request.form.get('password_strength'),
            'security_updates': request.form.get('security_updates'),
            'public_wifi_risk': request.form.get('public_wifi_risk'),
            'public_wifi_behavior': request.form.get('public_wifi_behavior'),
            'incident_reporting': request.form.get('incident_reporting')
        }
        return redirect('/attitudes')
    
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Security Behaviors - UCC Research</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>{BASE_STYLES}</style>
    </head>
    <body>
        <div class="container">
            <div class="progress-container">
                <div class="progress-stats">
                    <span>Step 6 of 6</span>
                    <span>Section E: Security Behaviors</span>
                </div>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: 85%"></div>
                </div>
            </div>
            
            <div class="card">
                <div class="card-header">
                    <h1>🛡️ Section E: Security Behaviors</h1>
                    <p>Tell us about your security practices</p>
                </div>
                
                <div class="card-content">
                    <form method="POST">
                        <div class="form-group">
                            <label><span class="label-highlight">28</span> Ever entered credentials after clicking a link? *</label>
                            <select name="entered_creds" required>
                                <option value="">-- Select --</option>
                                <option value="Yes">Yes</option>
                                <option value="No">No</option>
                                <option value="Not sure">Not sure</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">29</span> Experienced financial loss from online activity? *</label>
                            <select name="financial_loss" required>
                                <option value="">-- Select --</option>
                                <option value="Yes">Yes</option>
                                <option value="No">No</option>
                                <option value="Prefer not to say">Prefer not to say</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">30</span> Do you use Two-Factor Authentication (2FA)? *</label>
                            <select name="two_factor" required>
                                <option value="">-- Select --</option>
                                <option value="Always">Always when available</option>
                                <option value="Sometimes">Sometimes</option>
                                <option value="Never">Never</option>
                                <option value="Don't know">Don't know what it is</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">31</span> Do you use a password manager? *</label>
                            <select name="password_manager" required>
                                <option value="">-- Select --</option>
                                <option value="Yes">Yes</option>
                                <option value="No">No</option>
                                <option value="What's that">What's that?</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">32</span> Do you reuse passwords across sites? *</label>
                            <select name="password_reuse" required>
                                <option value="">-- Select --</option>
                                <option value="Yes, frequently">Yes, frequently</option>
                                <option value="Sometimes">Sometimes</option>
                                <option value="Rarely">Rarely</option>
                                <option value="Never">Never</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">33</span> Aware of public WiFi risks? *</label>
                            <select name="public_wifi_risk" required>
                                <option value="">-- Select --</option>
                                <option value="Yes">Yes</option>
                                <option value="Somewhat">Somewhat</option>
                                <option value="No">No</option>
                            </select>
                        </div>
                        
                        <button class="btn btn-primary btn-large" type="submit">Continue to Final Section →</button>
                    </form>
                </div>
            </div>
        </div>
    </body>
    </html>
    '''

# Attitudes Section
@app.route('/attitudes', methods=['GET', 'POST'])
def attitudes():
    if request.method == 'POST':
        all_data = {}
        for key in ['demo', 'digital', 'knowledge', 'behavior', 'security']:
            if key in session:
                all_data.update(session[key])
        
        skipped_sections = session.get('skipped_sections', [])
        scores = calculate_advanced_scores(all_data, skipped_sections)
        
        # Get completion time
        start_time = datetime.fromisoformat(session['start_time'])
        completion_time = (datetime.now() - start_time).total_seconds()
        
        # Get client IP hash (for anonymity)
        ip_hash = hashlib.sha256(request.remote_addr.encode()).hexdigest()[:16] if request.remote_addr else 'unknown'
        
        # Classify STEM
        stem_group, stem_confidence = classify_stem(
            session.get('demo', {}).get('program', ''),
            session.get('demo', {}).get('college', '')
        )
        
        # Prepare row data
        row = [
            # Metadata
            datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            session.get('response_id', ''),
            completion_time,
            ip_hash,
            
            # Section A
            session.get('demo', {}).get('gender', ''),
            session.get('demo', {}).get('age', ''),
            session.get('demo', {}).get('level', ''),
            session.get('demo', {}).get('college', ''),
            session.get('demo', {}).get('program', ''),
            stem_group,
            session.get('demo', {}).get('stem_confidence', ''),
            
            # Section B
            session.get('digital', {}).get('email_freq', ''),
            session.get('digital', {}).get('email_count', ''),
            session.get('digital', {}).get('email_provider', ''),
            session.get('digital', {}).get('banking_freq', ''),
            session.get('digital', {}).get('mobile_money', ''),
            session.get('digital', {}).get('shopping_freq', ''),
            session.get('digital', {}).get('social_count', ''),
            session.get('digital', {}).get('primary_social', ''),
            session.get('digital', {}).get('internet_years', ''),
            session.get('digital', {}).get('suspicious_msgs', ''),
            session.get('digital', {}).get('suspicious_freq', ''),
            session.get('digital', {}).get('suspicious_channels', ''),
            session.get('digital', {}).get('training_received', ''),
            session.get('digital', {}).get('training_types', ''),
            session.get('digital', {}).get('digital_literacy', ''),
            
            # Section C
            session.get('knowledge', {}).get('familiar', ''),
            session.get('knowledge', {}).get('phishing_def', ''),
            str(session.get('knowledge', {}).get('indicators', [])),
            session.get('knowledge', {}).get('password_stmt', ''),
            session.get('knowledge', {}).get('url_hovering', ''),
            session.get('knowledge', {}).get('https_knowledge', ''),
            session.get('knowledge', {}).get('padlock_knowledge', ''),
            session.get('knowledge', {}).get('email_header_knowledge', ''),
            session.get('knowledge', {}).get('spear_phishing', ''),
            session.get('knowledge', {}).get('smishing', ''),
            session.get('knowledge', {}).get('vishing', ''),
            session.get('knowledge', {}).get('phishing_stats', ''),
            session.get('knowledge', {}).get('ucc_reports', ''),
            scores['knowledge_score'],
            scores['knowledge_percentage'],
            
            # Section D
            session.get('behavior', {}).get('clicked_link', ''),
            session.get('behavior', {}).get('clicked_freq', ''),
            session.get('behavior', {}).get('clicked_context', ''),
            session.get('behavior', {}).get('ucc_scenario', ''),
            session.get('behavior', {}).get('bank_scenario', ''),
            session.get('behavior', {}).get('social_scenario', ''),
            session.get('behavior', {}).get('prize_scenario', ''),
            session.get('behavior', {}).get('urgent_scenario', ''),
            session.get('behavior', {}).get('familiar_scenario', ''),
            session.get('behavior', {}).get('scenario_rationale', ''),
            session.get('behavior', {}).get('confidence', ''),
            session.get('behavior', {}).get('report_knowledge', ''),
            session.get('behavior', {}).get('report_willingness', ''),
            scores['behavior_score'],
            scores['behavior_percentage'],
            
            # Section E
            session.get('security', {}).get('entered_creds', ''),
            session.get('security', {}).get('cred_context', ''),
            session.get('security', {}).get('financial_loss', ''),
            session.get('security', {}).get('loss_amount', ''),
            session.get('security', {}).get('compromise', ''),
            session.get('security', {}).get('compromise_type', ''),
            session.get('security', {}).get('two_factor', ''),
            session.get('security', {}).get('two_factor_barriers', ''),
            session.get('security', {}).get('password_manager', ''),
            session.get('security', {}).get('password_reuse', ''),
            session.get('security', {}).get('password_strength', ''),
            session.get('security', {}).get('security_updates', ''),
            session.get('security', {}).get('device_security', ''),
            session.get('security', {}).get('public_wifi_risk', ''),
            session.get('security', {}).get('public_wifi_behavior', ''),
            session.get('security', {}).get('incident_reporting', ''),
            session.get('security', {}).get('recovery_knowledge', ''),
            
            # Section F
            request.form.get('tech_opinion', ''),
            request.form.get('tech_opinion_rationale', ''),
            request.form.get('cyber_attitude', ''),
            request.form.get('perceived_vulnerability', ''),
            request.form.get('perceived_severity', ''),
            request.form.get('response_efficacy', ''),
            request.form.get('self_efficacy', ''),
            request.form.get('training_compulsory', ''),
            request.form.get('training_frequency', ''),
            request.form.get('training_format', ''),
            request.form.get('most_effective', ''),
            request.form.get('ucc_adequacy', ''),
            request.form.get('improvement_suggestions', ''),
            request.form.get('future_participation', ''),
            request.form.get('advanced_training', ''),
            
            # Advanced Metrics
            scores['abg_index'],
            scores['abg_interpretation'],
            scores['total_susceptibility'],
            scores['susceptibility_level'],
            stem_confidence,
            scores['knowledge_score'],
            scores['behavior_score'],
            0,  # attitude score placeholder
            ','.join(skipped_sections) if skipped_sections else 'None',
            '',  # incomplete sections
            '2.0'  # survey version
        ]
        
        save_to_csv(row)
        
        # Store scores for thank you page
        session['scores'] = scores
        session['stem_group'] = stem_group
        session.clear()
        session['scores'] = scores
        session['stem_group'] = stem_group
        
        return redirect('/thankyou')
    
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Attitudes - UCC Research</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>{BASE_STYLES}</style>
    </head>
    <body>
        <div class="container">
            <div class="progress-container">
                <div class="progress-stats">
                    <span>Final Section</span>
                    <span>Section F: Attitudes</span>
                </div>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: 95%"></div>
                </div>
            </div>
            
            <div class="card">
                <div class="card-header">
                    <h1>💭 Section F: Attitudes & Recommendations</h1>
                    <p>Your opinion matters for improving UCC cybersecurity</p>
                </div>
                
                <div class="card-content">
                    <form method="POST">
                        <div class="form-group">
                            <label><span class="label-highlight">34</span> "STEM students are less likely to fall for phishing" *</label>
                            <select name="tech_opinion" required>
                                <option value="">-- Select --</option>
                                <option value="Strongly agree">Strongly agree</option>
                                <option value="Agree">Agree</option>
                                <option value="Neutral">Neutral</option>
                                <option value="Disagree">Disagree</option>
                                <option value="Strongly disagree">Strongly disagree</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">35</span> Should cybersecurity training be compulsory at UCC? *</label>
                            <select name="training_compulsory" required>
                                <option value="">-- Select --</option>
                                <option value="Yes, with refreshers">Yes, with regular refreshers</option>
                                <option value="Yes, one-time">Yes, one-time orientation</option>
                                <option value="No, optional">No, optional only</option>
                                <option value="Not sure">Not sure</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">36</span> Most effective for preventing phishing? *</label>
                            <select name="most_effective" required>
                                <option value="">-- Select --</option>
                                <option value="Training workshops">Training workshops</option>
                                <option value="Email warnings">Real-time warnings</option>
                                <option value="Filtering systems">Better filtering</option>
                                <option value="Awareness campaigns">Awareness campaigns</option>
                                <option value="All combined">All combined</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">37</span> Rate UCC's current security awareness: *</label>
                            <select name="ucc_adequacy" required>
                                <option value="">-- Select --</option>
                                <option value="Excellent">Excellent</option>
                                <option value="Good">Good</option>
                                <option value="Fair">Fair</option>
                                <option value="Poor">Poor</option>
                                <option value="Don't know">Don't know</option>
                            </select>
                        </div>
                        
                        <div class="form-group">
                            <label><span class="label-highlight">38</span> Suggestions for improvement (optional)</label>
                            <textarea name="improvement_suggestions" rows="4" placeholder="Your ideas to improve cybersecurity at UCC..."></textarea>
                        </div>
                        
                        <button class="btn btn-success btn-large" type="submit">✓ SUBMIT SURVEY</button>
                    </form>
                </div>
            </div>
        </div>
    </body>
    </html>
    '''

@app.route('/thankyou')
def thankyou():
    scores = session.get('scores', {})
    stem_group = session.get('stem_group', 'Participant')
    
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Thank You - UCC Research</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>{BASE_STYLES}</style>
        <style>
            .result-card {{
                background: linear-gradient(135deg, #f6f9fc 0%, #e6f0f9 100%);
                border-radius: 20px;
                padding: 30px;
                margin: 20px 0;
                text-align: center;
            }}
            .score-badge {{
                display: inline-block;
                padding: 15px 30px;
                border-radius: 50px;
                font-size: 1.2rem;
                font-weight: bold;
                margin: 10px;
            }}
            .level-low {{ background: #d4edda; color: #155724; }}
            .level-moderate {{ background: #fff3cd; color: #856404; }}
            .level-high {{ background: #f8d7da; color: #721c24; }}
            .level-critical {{ background: #dc3545; color: white; }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="card">
                <div class="card-header">
                    <h1>✨ Thank You for Participating!</h1>
                    <p>Your contribution to UCC cybersecurity research is invaluable</p>
                </div>
                
                <div class="card-content" style="text-align: center;">
                    <div style="font-size: 5rem; margin: 20px 0;">🎉</div>
                    
                    <h2>Your Response Has Been Recorded</h2>
                    <p style="color: #666; margin: 20px 0;">Your anonymous data will help improve cybersecurity awareness at UCC.</p>
                    
                    <div class="result-card">
                        <h3>Your Cybersecurity Profile</h3>
                        <div>
                            <div class="score-badge level-{'low' if scores.get('susceptibility_level') == 'Low' else 'moderate' if scores.get('susceptibility_level') == 'Moderate' else 'high' if scores.get('susceptibility_level') == 'High' else 'critical'}">
                                Susceptibility Level: {scores.get('susceptibility_level', 'N/A')}
                            </div>
                        </div>
                        <p>Knowledge Score: {scores.get('knowledge_score', 0):.1f}% | Behavior Score: {scores.get('behavior_score', 0):.1f}%</p>
                        <p>ABG Index: {scores.get('abg_index', 0):.1f} - {scores.get('abg_interpretation', '')}</p>
                    </div>
                    
                    <div class="info-grid" style="margin: 40px 0;">
                        <div class="info-card">
                            <div class="info-icon">🔒</div>
                            <h4>Tip 1</h4>
                            <p>Never click links in unsolicited messages</p>
                        </div>
                        <div class="info-card">
                            <div class="info-icon">🔑</div>
                            <h4>Tip 2</h4>
                            <p>Enable Two-Factor Authentication (2FA)</p>
                        </div>
                        <div class="info-card">
                            <div class="info-icon">🛡️</div>
                            <h4>Tip 3</h4>
                            <p>Verify before trusting - visit official sites directly</p>
                        </div>
                    </div>
                    
                    <a href="/" class="btn btn-primary" style="margin: 20px;">← Return to Home</a>
                    
                    <div style="margin-top: 40px; color: #666;">
                        <p><strong>Research Team:</strong> BSc Information Technology Final Year Group Project</p>
                        <p><strong>Supervisor:</strong> Dr. Franklin Amoh, PhD</p>
                    </div>
                </div>
            </div>
        </div>
    </body>
    </html>
    '''

@app.route('/dashboard')
def dashboard():
    if not os.path.exists(CSV_FILE):
        return "No responses yet"
    
    with open(CSV_FILE, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        data = list(reader)
    
    if not data:
        return "No responses yet"
    
    total = len(data)
    stem_count = sum(1 for row in data if row.get('Student_Group_Classification') == 'STEM')
    non_stem_count = sum(1 for row in data if row.get('Student_Group_Classification') == 'Non-STEM')
    
    try:
        knowledge_scores = [float(row.get('Knowledge_Score', 0)) for row in data if row.get('Knowledge_Score')]
        behavior_scores = [float(row.get('Behavior_Score', 0)) for row in data if row.get('Behavior_Score')]
        avg_knowledge = sum(knowledge_scores) / len(knowledge_scores) if knowledge_scores else 0
        avg_behavior = sum(behavior_scores) / len(behavior_scores) if behavior_scores else 0
    except:
        avg_knowledge = avg_behavior = 0
    
    html = f'''
    <!DOCTYPE html>
    <html>
    <head>
        <title>Dashboard - UCC Research</title>
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
        <style>
            body {{
                font-family: 'Poppins', sans-serif;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                min-height: 100vh;
                padding: 20px;
            }}
            .container {{
                max-width: 1200px;
                margin: 0 auto;
            }}
            .card {{
                background: white;
                border-radius: 20px;
                padding: 30px;
                margin: 20px 0;
                box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            }}
            h1, h2 {{ color: #0c2461; }}
            .stats-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                gap: 20px;
                margin: 30px 0;
            }}
            .stat-box {{
                background: #f8fafc;
                padding: 20px;
                border-radius: 15px;
                text-align: center;
            }}
            .stat-value {{
                font-size: 2.5rem;
                font-weight: bold;
                color: #667eea;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th, td {{
                padding: 12px;
                text-align: left;
                border-bottom: 1px solid #ddd;
            }}
            th {{
                background: #667eea;
                color: white;
            }}
            tr:hover {{
                background: #f5f5f5;
            }}
            .btn {{
                padding: 10px 20px;
                background: #667eea;
                color: white;
                text-decoration: none;
                border-radius: 10px;
                display: inline-block;
                margin: 10px;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="card">
                <h1>📊 Research Dashboard</h1>
                <p>Real-time data collection statistics</p>
                
                <div class="stats-grid">
                    <div class="stat-box">
                        <div class="stat-value">{total}</div>
                        <div>Total Responses</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">{stem_count}</div>
                        <div>STEM Students</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">{non_stem_count}</div>
                        <div>Non-STEM Students</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">{avg_knowledge:.1f}%</div>
                        <div>Avg Knowledge</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">{avg_behavior:.1f}%</div>
                        <div>Avg Behavior</div>
                    </div>
                </div>
                
                <a href="/export" class="btn">📥 Download Data (CSV)</a>
                <a href="/" class="btn">🏠 Home</a>
                
                <h2>Recent Responses</h2>
                <table>
                    <tr>
                        <th>ID</th>
                        <th>Group</th>
                        <th>Knowledge</th>
                        <th>Behavior</th>
                        <th>Susceptibility</th>
                        <th>Time</th>
                    </tr>
    '''
    
    for row in data[-10:]:
        html += f'''
                    <tr>
                        <td>{row.get('Response_ID', '')[:8]}...</td>
                        <td>{row.get('Student_Group_Classification', '')}</td>
                        <td>{row.get('Knowledge_Score', '')}%</td>
                        <td>{row.get('Behavior_Score', '')}%</td>
                        <td>{row.get('Susceptibility_Level', '')}</td>
                        <td>{row.get('Timestamp', '')[:10]}</td>
                    </tr>
        '''
    
    html += '''
                </table>
            </div>
        </div>
    </body>
    </html>
    '''
    
    return html

@app.route('/export')
def export():
    if not os.path.exists(CSV_FILE):
        return "No data available"
    
    with open(CSV_FILE, 'r', encoding='utf-8') as f:
        csv_data = f.read()
    
    return Response(
        csv_data,
        mimetype="text/csv",
        headers={"Content-disposition": "attachment; filename=ucc_phishing_data.csv"}
    )

if __name__ == '__main__':
    print("=" * 70)
    print("🎓 UCC PHISHING SUSCEPTIBILITY STUDY - ENHANCED VERSION")
    print("=" * 70)
    print("✅ Research Team: BSc Information Technology Final Year Group Project")
    print("👨‍🏫 Supervisor: Dr. Franklin Amoh, PhD")
    print("=" * 70)
    print("📊 Survey Features:")
    print("   • 38+ comprehensive questions")
    print("   • 70+ data points per response")
    print("   • Advanced skip logic")
    print("   • Professional UI/UX with animations")
    print("   • Real-time scoring and ABG Index")
    print("   • Mobile responsive design")
    print("=" * 70)
    print("📈 Data Collection:")
    print(f"   • CSV Backup: {CSV_FILE}")
    print("   • Dashboard: http://localhost:5000/dashboard")
    print("   • Export: http://localhost:5000/export")
    print("=" * 70)
    print("🌐 Your Survey URL (when deployed):")
    print("   • PythonAnywhere: https://uccphishing2024.pythonanywhere.com")
    print("=" * 70)
    
    app.run(debug=False, host='0.0.0.0', port=5000, use_reloader=False)

🎓 UCC PHISHING SUSCEPTIBILITY STUDY - ENHANCED VERSION
✅ Research Team: BSc Information Technology Final Year Group Project
👨‍🏫 Supervisor: Dr. Franklin Amoh, PhD
📊 Survey Features:
   • 38+ comprehensive questions
   • 70+ data points per response
   • Advanced skip logic
   • Professional UI/UX with animations
   • Real-time scoring and ABG Index
   • Mobile responsive design
📈 Data Collection:
   • CSV Backup: phishing_survey_responses_comprehensive.csv
   • Dashboard: http://localhost:5000/dashboard
   • Export: http://localhost:5000/export
🌐 Your Survey URL (when deployed):
   • PythonAnywhere: https://uccphishing2024.pythonanywhere.com
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.77.48.46:5000
Press CTRL+C to quit
